# RAGAS Evaluation — Faithfulness and Answer Relevancy
## Measure Before You Ship — RAGAS Faithfulness and Relevancy
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/74-ragas-evaluation/ragas_evaluation_workbook.ipynb)

Evaluates a RAG pipeline using RAGAS metrics: faithfulness (does the answer stick to the retrieved context?) and answer_relevancy (does it address the question?). Scores a 5-question golden dataset.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why RAG metrics differ from accuracy; faithfulness vs relevancy |
| 2 | **Golden Dataset** — 5-row QA dataset with ground_truth and contexts |
| 3 | **RAG Pipeline** — Chroma retriever + LLM generates answers for each question |
| 4 | **RAGAS Evaluate** — `evaluate(dataset, metrics=[faithfulness, answer_relevancy])` |
| 5 | **Score Analysis** — Interpret scores; identify weak questions |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langchain-chroma", "chromadb", "ragas", "datasets", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Why RAG-Specific Metrics?

Standard accuracy metrics (exact match, F1) break down for RAG because the model can:
- **Hallucinate** — answer confidently with facts not in the retrieved context
- **Retrieve irrelevant chunks** — context looks related but doesn't actually help
- **Paraphrase correctly** — match the meaning without matching the words

RAGAS defines four metrics designed specifically for these failure modes:

| Metric | Question it answers | Needs ground truth? |
|--------|-------------------|---------------------|
| **Faithfulness** | Is the answer grounded in the retrieved context? | No |
| **Answer Relevancy** | Does the answer address the question? | No |
| **Context Precision** | Are retrieved chunks relevant to the question? | Yes |
| **Context Recall** | Did retrieval find everything needed for the answer? | Yes |

Faithfulness and Answer Relevancy are the two you can run without a labelled dataset — they use the LLM itself as a judge.

> **Prerequisite:** You've built RAG pipelines before (see examples 1, 10, 17, 25). This notebook focuses on *measuring* them, not building them.

In [ ]:
# Part 1 — install check and imports
# RAGAS uses the LLM as a judge internally — it makes additional API calls during evaluate()
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print("RAGAS metrics available:")
for m in [faithfulness, answer_relevancy, context_precision, context_recall]:
    print(f"  - {m.name}")

print()
print("faithfulness + answer_relevancy: no ground truth needed")
print("context_precision + context_recall: require ground_truth labels")

## Part 2 — Building the Golden Dataset

A RAGAS evaluation dataset has up to five fields per row:

| Field | Required for | Description |
|-------|-------------|-------------|
| `question` | all metrics | The user's question |
| `answer` | all metrics | The LLM's generated answer |
| `contexts` | all metrics | List of retrieved text chunks |
| `ground_truth` | context_recall, context_precision | The ideal reference answer |

The `contexts` field is a **list of strings** — one string per retrieved chunk. RAGAS checks whether the `answer` can be supported by those chunks (faithfulness) and whether the chunks are relevant to the question (context precision/recall).

### Building a RAG pipeline to generate answers

We use a Chroma vector store with 5 documents as the knowledge base, then run each question through retrieval + generation to produce the `answer` field.

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# tiny knowledge base — in production use RecursiveCharacterTextSplitter over real docs
DOCS = [
    Document(page_content="LangGraph is a library for building stateful multi-actor applications with LLMs."),
    Document(page_content="RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, and context recall."),
    Document(page_content="ChromaDB is an open-source embedding database for AI applications."),
    Document(page_content="Retrieval-augmented generation combines vector search with language model generation."),
    Document(page_content="LangSmith provides tracing and evaluation for LangChain and LangGraph applications."),
]

QUESTIONS = [
    {"q": "What is LangGraph?",
     "gt": "LangGraph is a library for building stateful multi-actor applications with LLMs.",
     "ctx": ["LangGraph is a library for building stateful multi-actor applications with LLMs."]},
    {"q": "What does RAGAS evaluate?",
     "gt": "RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, and context recall.",
     "ctx": ["RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, and context recall."]},
    {"q": "What is ChromaDB?",
     "gt": "ChromaDB is an open-source embedding database.",
     "ctx": ["ChromaDB is an open-source embedding database for AI applications."]},
    {"q": "What is retrieval-augmented generation?",
     "gt": "RAG combines vector search with language model generation.",
     "ctx": ["Retrieval-augmented generation combines vector search with language model generation."]},
    {"q": "What is LangSmith used for?",
     "gt": "LangSmith provides tracing and evaluation for LangChain applications.",
     "ctx": ["LangSmith provides tracing and evaluation for LangChain and LangGraph applications."]},
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm        = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
vs         = Chroma.from_documents(DOCS, embeddings, collection_name="ragas-eval")
retriever  = vs.as_retriever(search_kwargs={"k": 1})

# generate answers from the RAG pipeline
rows = []
for item in QUESTIONS:
    docs    = retriever.invoke(item["q"])
    ctx_str = docs[0].page_content if docs else ""
    prompt  = f"Context: {ctx_str}\n\nQuestion: {item['q']}\nAnswer briefly:"
    answer  = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    rows.append({"question": item["q"], "answer": answer,
                 "contexts": item["ctx"], "ground_truth": item["gt"]})
    print(f"Q: {item['q']}")
    print(f"A: {answer}")
    print()

print(f"Dataset ready: {len(rows)} rows")

## Part 3 — Running RAGAS Metrics

`evaluate()` takes a Hugging Face `Dataset` and a list of metrics. It makes LLM calls internally to judge faithfulness and relevancy — expect a few extra API calls per row.

### Faithfulness — is the answer grounded?

RAGAS breaks the answer into atomic claims and checks each one against the context. Score = fraction of claims supported by context. A score of 1.0 means every claim in the answer can be traced back to retrieved text.

### Answer Relevancy — does it address the question?

RAGAS generates synthetic questions from the answer and measures how similar they are to the original question. High relevancy = the answer stays on topic. Low relevancy = the answer drifts or is too generic.

### Context Precision — are retrieved chunks useful?

Measures what fraction of retrieved chunks are actually relevant to answering the question. Requires `ground_truth`.

### Context Recall — did retrieval find enough?

Measures how much of the ground-truth answer is covered by the retrieved chunks. Requires `ground_truth`.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# convert rows list to HuggingFace Dataset — RAGAS requires this format
dataset = Dataset.from_list(rows)

print("Running RAGAS evaluation (this makes extra LLM calls internally)...")
result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
                  llm=ragas_llm, embeddings=ragas_embeddings)

def metric_mean(evaluation_result, metric_name):
    scores = evaluation_result.to_pandas()[metric_name].dropna()
    return float(scores.mean()) if len(scores) else 0.0

print("\nRAGAS Scores:")
print(f"  Faithfulness:      {metric_mean(result, 'faithfulness'):.3f}  (> 0.8 = answers grounded in context)")
print(f"  Answer Relevancy:  {metric_mean(result, 'answer_relevancy'):.3f}  (> 0.8 = answers address the question)")
print(f"  Context Precision: {metric_mean(result, 'context_precision'):.3f}  (> 0.8 = retrieved chunks are relevant)")
print(f"  Context Recall:    {metric_mean(result, 'context_recall'):.3f}  (> 0.8 = retrieval found what was needed)")

## Part 4 — Per-Row Analysis and Threshold Reporting

Aggregate scores hide per-question problems. A faithfulness of 0.80 could mean every question scored 0.80, or that four questions scored 1.0 and one scored 0.0 — very different situations.

RAGAS returns a `DatasetResult` that you can convert to a pandas DataFrame to inspect per-row scores and flag rows that fall below threshold.

This is the core of a **regression testing workflow**:
1. Run RAGAS on a golden dataset after every model or prompt change
2. Flag any row that drops below threshold
3. Investigate the flagged rows — check if retrieval failed, or if the LLM hallucinated

In [ ]:
# per-row score breakdown — convert result to DataFrame for inspection
df = result.to_pandas()

THRESHOLD = 0.75

print("Per-question scores:")
print(f"{'Question':<45} {'Faith':>6} {'Relev':>6} {'CPrec':>6} {'CRec':>6} {'Flag'}")
print("-" * 80)
for _, row in df.iterrows():
    faith = row.get('faithfulness',      0.0) or 0.0
    relev = row.get('answer_relevancy',  0.0) or 0.0
    cprec = row.get('context_precision', 0.0) or 0.0
    crec  = row.get('context_recall',    0.0) or 0.0
    flag  = "WARN" if any(v < THRESHOLD for v in [faith, relev, cprec, crec]) else "ok"
    question = row.get('question', row.get('user_input', ''))
    q_short = question[:42] + "..." if len(question) > 42 else question
    print(f"{q_short:<45} {faith:>6.2f} {relev:>6.2f} {cprec:>6.2f} {crec:>6.2f}  {flag}")

flagged = df[
    (df.get('faithfulness',      0) < THRESHOLD) |
    (df.get('answer_relevancy',  0) < THRESHOLD) |
    (df.get('context_precision', 0) < THRESHOLD) |
    (df.get('context_recall',    0) < THRESHOLD)
]
print(f"\nFlagged rows (any score < {THRESHOLD}): {len(flagged)}/{len(df)}")
if not flagged.empty:
    print("  Investigate these questions — retrieval or generation may be failing.")

## Part 5 — Custom Scorer

Sometimes RAGAS's built-in metrics don't fit your domain. You can write a custom scorer as a plain Python function and combine it with RAGAS metrics.

A common custom metric: **answer length score** — penalise answers that are too short (incomplete) or too long (verbose). Not a substitute for semantic metrics, but useful as a fast sanity check on the pipeline's verbosity.

More practically, you can combine RAGAS scores with a custom **LLM-as-judge** function that uses a rubric specific to your use case (e.g. medical accuracy, tone, citation format).

In [ ]:
import numpy as np

# --- custom scorer: length-normalised completeness proxy ---
def length_score(answer: str, min_words: int = 5, max_words: int = 80) -> float:
    # returns 1.0 if word count is in [min_words, max_words], scales down outside that range
    n = len(answer.split())
    if n < min_words:
        return round(n / min_words, 3)
    if n > max_words:
        return round(max_words / n, 3)
    return 1.0

# --- apply custom scorer alongside RAGAS scores ---
print("Custom length scores per question:")
ragas_df = result.to_pandas()
for i, row_item in enumerate(rows):
    ls = length_score(row_item["answer"])
    faith = ragas_df.iloc[i].get("faithfulness", 0.0) or 0.0
    print(f"  Q: {row_item['question'][:50]}")
    print(f"     length_score={ls:.2f}  faithfulness={faith:.2f}")

# --- simple composite score ---
print("\nComposite score (0.5 * faithfulness + 0.3 * answer_relevancy + 0.2 * length):")
for i, row_item in enumerate(rows):
    r = ragas_df.iloc[i]
    faith = r.get("faithfulness",     0.0) or 0.0
    relev = r.get("answer_relevancy", 0.0) or 0.0
    ls    = length_score(row_item["answer"])
    comp  = 0.5 * faith + 0.3 * relev + 0.2 * ls
    print(f"  {row_item['question'][:45]:<45}  composite={comp:.3f}")

## Exercises

Work through these in order — each builds on the sections above.

---

**Exercise 1 — Add a Sixth Question**

Add a new question to `QUESTIONS` about ChromaDB or LangSmith. Re-run the pipeline to generate its answer and re-run `evaluate()`. Does the new row drag down or improve the average scores?

---

**Exercise 2 — Lower k and Measure the Impact**

Change the retriever to `search_kwargs={"k": 3}` (retrieve 3 chunks instead of 1). Update `contexts` in each row to include all 3 chunks. Re-run RAGAS and compare `context_recall` — does more context help?

---

**Exercise 3 — Faithfulness Drop Test**

Manually write one answer that clearly **hallucinates** — includes a fact not in any retrieved context. Add it to the dataset and observe the faithfulness score for that row. How low does it go?

---

**Exercise 4 — Custom LLM Judge**

Write a function `llm_judge(question: str, answer: str) -> float` that asks `gpt-4o-mini` to rate the answer quality from 0 to 10 and returns the score normalised to 0–1. Apply it to all rows and compute the correlation with RAGAS faithfulness.

In [ ]:
# Exercise 1 — Add a Sixth Question
# Add your new row to QUESTIONS, re-run the pipeline cell (code-p2), then re-run evaluate()
NEW_QUESTION = {
    "q": "What does ChromaDB store?",
    "gt": "ChromaDB stores embeddings for AI applications.",
    "ctx": ["ChromaDB is an open-source embedding database for AI applications."],
}
# QUESTIONS.append(NEW_QUESTION)
# Then re-run the pipeline and evaluate cells above
print("Uncomment the line above, then re-run code-p2 and code-p3.")

In [ ]:
# Exercise 4 — Custom LLM Judge (starter)
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

_judge_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

def llm_judge(question: str, answer: str) -> float:
    # TODO: ask the LLM to rate answer quality 0-10, parse the number, return /10
    pass

# test it on one row
# score = llm_judge(rows[0]["question"], rows[0]["answer"])
# print(f"Judge score: {score}")

---
## Answer Key

Attempt the exercises before reading the answers below.

In [ ]:
# Answer 2 — Lower k=3 and Measure Impact
# Change retriever to k=3, rebuild contexts list, re-evaluate

retriever_k3 = vs.as_retriever(search_kwargs={"k": 3})

rows_k3 = []
for item in QUESTIONS:
    docs    = retriever_k3.invoke(item["q"])
    ctx_chunks = [d.page_content for d in docs]
    ctx_str    = "\n".join(ctx_chunks)
    prompt     = f"Context: {ctx_str}\n\nQuestion: {item['q']}\nAnswer briefly:"
    answer     = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    rows_k3.append({"question": item["q"], "answer": answer,
                    "contexts": ctx_chunks, "ground_truth": item["gt"]})

dataset_k3 = Dataset.from_list(rows_k3)
result_k3  = evaluate(dataset_k3, metrics=[faithfulness, answer_relevancy, context_recall],
                      llm=ragas_llm, embeddings=ragas_embeddings)
print(f"k=1  context_recall: {metric_mean(result, 'context_recall'):.3f}")
print(f"k=3  context_recall: {metric_mean(result_k3, 'context_recall'):.3f}")
print("More context chunks generally improves recall but may hurt precision.")

In [ ]:
# Answer 3 — Faithfulness Drop Test
# Insert a hallucinated answer and observe the score

hallucinated_row = {
    "question":     "What is LangGraph?",
    "answer":       "LangGraph is a cloud database made by Google that stores vectors in BigQuery.",
    "contexts":     ["LangGraph is a library for building stateful multi-actor applications with LLMs."],
    "ground_truth": "LangGraph is a library for building stateful multi-actor applications with LLMs.",
}
hallucinated_ds = Dataset.from_list([hallucinated_row])
hall_result = evaluate(hallucinated_ds, metrics=[faithfulness], llm=ragas_llm)
print(f"Faithfulness with hallucinated answer: {metric_mean(hall_result, 'faithfulness'):.3f}")
print("Expected: very low (< 0.3) — the answer contradicts the context entirely.")

In [ ]:
# Answer 4 — Custom LLM Judge
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import re

_judge = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

def llm_judge(question: str, answer: str) -> float:
    prompt = (
        f"Rate this answer's quality from 0 to 10.\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n"
        f"Respond with just a number 0-10."
    )
    response = _judge.invoke([HumanMessage(content=prompt)]).content.strip()
    match = re.search(r'\d+(\.\d+)?', response)
    return round(float(match.group()) / 10, 3) if match else 0.0

print("LLM Judge scores vs RAGAS Faithfulness:")
ragas_scores = result.to_pandas()["faithfulness"].tolist()
for i, row_item in enumerate(rows):
    judge = llm_judge(row_item["question"], row_item["answer"])
    faith = ragas_scores[i] if i < len(ragas_scores) else 0.0
    print(f"  Q: {row_item['question'][:45]:<45}  judge={judge:.2f}  faithfulness={faith:.2f}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*